# OpenMIC classifier training (Colab)

**Before you run:** Runtime → GPU. Push latest repo to GitHub. Drive must contain:

```text
MyDrive/Decomposition/data/openmic/
├── openmic-2018/   (audio/, openmic-2018.npz, partitions/split01_*.csv)
└── mel_cache/      (empty OK)
```

**Ordering:** Cell 3 mounts Drive, then imports `src.config`. Once imported, paths are locked for the session — Runtime → Restart to reload.

## Cell 1 — Clone repo

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/Decomposition")
REPO_URL = "https://github.com/alexnguyen25/Decomposition.git"

if REPO_DIR.exists():
    %cd /content/Decomposition
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd /content/Decomposition

sys.path.insert(0, str(REPO_DIR))
print("cwd:", os.getcwd())

## Cell 2 — Install dependencies

Colab already has CUDA PyTorch + numpy. `feature_extraction` needs `librosa` (and `soundfile` as a common audio backend). Skip Demucs / Essentia.

In [ ]:
!pip install -q librosa soundfile

import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## Cell 3 — Mount Drive + path sanity

Mount first, then import `src.config` (first `src` import in the notebook). Asserts use config paths / `TRAIN_PARTITION` so Drive layout stays a single source of truth.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# First src import this session — /content check + Drive paths get baked in here.
# Re-running this cell later does NOT refresh config's paths (Python caches the import);
# if paths need to change, restart the runtime.import src.config as config

assert config.OPENMIC_DIR.is_dir(), f"Missing OpenMIC dir: {config.OPENMIC_DIR}"
assert (config.OPENMIC_DIR / "openmic-2018.npz").exists(), "Missing openmic-2018.npz"
assert (config.OPENMIC_DIR / "partitions" / config.TRAIN_PARTITION).exists(), (
    f"Missing partition: {config.TRAIN_PARTITION}"
)

config.CACHE_DIR.mkdir(parents=True, exist_ok=True)
config.CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("OPENMIC_DIR:", config.OPENMIC_DIR)
print("CACHE_DIR:", config.CACHE_DIR)
print("CHECKPOINT_PATH:", config.CHECKPOINT_PATH)
print("TRAIN_PARTITION:", config.TRAIN_PARTITION)
print("Cached mels so far:", len(list(config.CACHE_DIR.glob("*.npy"))))

## Cell 4 — Precompute mel cache

`src.config` is already loaded from Cell 3. Re-runs skip existing `.npy` files.

In [ ]:
import precompute_mel
from src.config import CACHE_DIR

precompute_mel.main()
print("cached .npy files:", len(list(CACHE_DIR.glob("*.npy"))))

## Cell 5 — Train

In [ ]:
import src.classification.train as train_mod
from src.config import CHECKPOINT_PATH

train_mod.main()
print("checkpoint exists:", CHECKPOINT_PATH.exists())
print("checkpoint path:", CHECKPOINT_PATH)